In [1]:
import json
from itertools import islice
from pprint import pprint

import textwrap

In [2]:
def get_line(path, n):
    """Return line n (1-based) of a JSONL file, parsed."""
    with open(path, encoding="utf-8") as f:
        line = next(islice(f, n - 1, n), None)
    if line is None:
        raise IndexError(f"{path} has fewer than {n} lines")
    return json.loads(line)

def get_lines(path, n, before=1, after=1):
    start = max(1, n - before)
    with open(path, encoding="utf-8") as f:
        return [(i, json.loads(l)) for i, l in enumerate(islice(f, start - 1, n + after), start)]

def show(obj, indent=0, max_chars=None):
    """Print a JSON-like value; multi-line strings are printed as real text."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)) or (isinstance(v, str) and "\n" in v):
                print(f"{pad}{k}:")
                show(v, indent + 1, max_chars)
            else:
                print(f"{pad}{k}: {v!r}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            print(f"{pad}[{i}]")
            show(v, indent + 1, max_chars)
    elif isinstance(obj, str):
        text = obj if max_chars is None or len(obj) <= max_chars else obj[:max_chars] + f" …(+{len(obj) - max_chars})"
        print(textwrap.indent(text, pad))
    else:
        print(f"{pad}{obj!r}")

In [3]:
rec = get_line("data/cli/2026-10-12.jsonl", 2)
print(f"--- structure ---")
pprint(rec["request"]["body"]["messages"], sort_dicts=False, depth=2)

--- structure ---
[{'role': 'user', 'content': [...]}]


In [4]:
print(f"--- content ---")
for msg in rec["request"]["body"]["messages"]:
    show(msg["content"], max_chars=200)

--- content ---
[0]
  type: 'text'
  text:
    <session>
    what is the content of the current dir ?
    </session>

    Write the title in the predominant language of the session — a stray word or code token in another language doesn't change it, and neithe …(+41)


In [15]:
pprint(rec["request"]["body"]["system"], sort_dicts=False, depth=2)

[{'type': 'text',
  'text': 'x-anthropic-billing-header: cc_version=2.1.296.a99; '
          'cc_entrypoint=cli;'},
 {'type': 'text',
  'text': "You are Claude Code, Anthropic's official CLI for Claude.",
  'cache_control': {...}},
 {'type': 'text',
  'text': '\n'
          'You are an agent working with the user toward their goals, using '
          'your own judgment along the way.\n'
          '\n'
          'IMPORTANT: Assist with authorized security testing, defensive '
          'security, CTF challenges, and educational contexts. Refuse requests '
          'for destructive techniques, DoS attacks, mass targeting, supply '
          'chain compromise, or detection evasion for malicious purposes. '
          'Dual-use security tools (C2 frameworks, credential testing, exploit '
          'development) require clear authorization context: pentesting '
          'engagements, CTF competitions, security research, or defensive use '
          'cases.\n'
          '\n'
          '# Ha

In [5]:
print(f"--- structure ---")
pprint(rec["response"]["body"]["content"], sort_dicts=False, depth=2)
print(f"--- content ---")
for msg in rec["response"]["body"]["content"]:
    show(msg, max_chars=300)

--- structure ---
[{'type': 'text', 'text': '{"title":"Current directory contents"}'}]
--- content ---
type: 'text'
text: '{"title":"Current directory contents"}'


In [16]:
rec = get_line("data/cli/2026-10-12.jsonl", 3)
print(f"--- structure ---")
pprint(rec["request"]["body"]["messages"], sort_dicts=False, depth=3)

--- structure ---
[{'role': 'user', 'content': [{...}, {...}, {...}]},
 {'role': 'system', 'content': [{...}], 'output_config': {'effort': 'medium'}}]


In [7]:
print(f"--- content ---")
for msg in rec["request"]["body"]["messages"]:
    show(msg["content"], max_chars=200)

--- content ---
[0]
  type: 'text'
  text:
    <system-reminder>
    As you answer the user's questions, you can use the following context:
    # userEmail
    The user's email address is wfauriat@gmail.com. Use it only to identify the user, such as for autho …(+360)
[1]
  type: 'text'
  text:
    <system-reminder>
    Attribution for git commits and pull requests you create from here on (this replaces Claude Code's own earlier attribution guidance, such as a previous copy of this reminder; the use …(+380)
[2]
  type: 'text'
  text: 'what is the content of the current dir ?'
[0]
  type: 'text'
  text:
    # Environment
    You have been invoked in the following environment: 
     - Primary working directory: /home/ai-user/Documents/Sandbox/LLMgrab
     - Is a git repository: false
     - Platform: linux
     - Shell: bash …(+23525)
  cache_control:
    type: 'ephemeral'
    ttl: '1h'


In [8]:
print(f"--- structure ---")
pprint(rec["response"]["body"]["content"], sort_dicts=False, depth=2)
print(f"--- content ---")
for msg in rec["response"]["body"]["content"]:
    show(msg, max_chars=300)

--- structure ---
[{'type': 'tool_use',
  'id': 'toolu_01FJEnTwwVyS3AwZwzRP63L8',
  'name': 'Bash',
  'input': {...},
  'caller': {...}}]
--- content ---
type: 'tool_use'
id: 'toolu_01FJEnTwwVyS3AwZwzRP63L8'
name: 'Bash'
input:
  command: 'ls -la /home/ai-user/Documents/Sandbox/LLMgrab'
  description: 'List current directory contents'
caller:
  type: 'direct'


In [9]:
rec = get_line("data/cli/2026-10-12.jsonl", 4)
print(f"--- structure ---")
pprint(rec["request"]["body"]["messages"], sort_dicts=False, depth=3)

--- structure ---
[{'role': 'user', 'content': [{...}, {...}, {...}]},
 {'role': 'system',
  'content': '# Environment\n'
             'You have been invoked in the following environment: \n'
             ' - Primary working directory: '
             '/home/ai-user/Documents/Sandbox/LLMgrab\n'
             ' - Is a git repository: false\n'
             ' - Platform: linux\n'
             ' - Shell: bash\n'
             ' - OS Version: Linux 7.0.0-38-generic\n'
             ' - Scratchpad directory: '
             '/tmp/claude-1001/-home-ai-user-Documents-Sandbox-LLMgrab/8dc6a9b7-e607-44a8-b3bf-efa61235f3d8/scratchpad '
             '— always use it for temporary files (intermediate results, '
             "scripts, outputs that don't belong in the project) instead of "
             '`/tmp` or other system temp directories; it is session-specific, '
             'isolated from the project, and can generally be used without '
             'permission prompts. Only use `/tmp` if the user 

In [10]:
print(f"--- structure ---")
pprint(rec["response"]["body"]["content"], sort_dicts=False, depth=2)
print(f"--- content ---")
for msg in rec["response"]["body"]["content"]:
    show(msg, max_chars=300)

--- structure ---
[{'type': 'thinking',
  'thinking': '',
  'signature': 'CAQSogUKEAgSGAI4AUIIdGhpbmtpbmcSDGYsLrg4APNwLq9qrhoM80zlNNd68wyDGP3oIjBnZIdz2aomVr0KYxrYzCZK2KNeLUxsft/uTy+G6DbjINKMCA2RAh9NLX/OvOziTfcqvwQil4NORqLXXOA2TevINMl4958SVXBGy/xVI6+fHtLeUCsXXNffxP3nDCdeN2cnaWINS2bObsJV/L5fEFL9OcAZ8Hu3UrBI5olzLELNJa+UZUScj1/QgSrpl+qi1Gmg/9KhZS+eFZH+ONlBy7MrEwdYiGoukTVT/CCqz2gLIQx4z3PP5lKW7uuyC9mpuJmHmv7Nx9z1OglpQ6JngB0R5Ct02I/b4YkYfBwC36se1a9xCDGcWxqOA4osbhRdnsita/GPtkTXIC/mqNpzJIc9FwJgYCBhCEZXbfaTDrTjY0TSII/OzHezbmXbZQc9xD4xIim8nWPNl+Ws/Zc2r8NR4ffHIZST7GDT02JjX9HlEshIpANK3pNuMSZfoZlA0lppmEHE60+Xbo7hUS/I+OeEde43BdaOQKcapsB1miZnxmWsoxkmmTYAWIjeAsupmSOYhrMF65O7IbfZciRL1iqNzXQw+Hwc+6IWJdCADR1vUU6CJVzh0KIH0RamLMcg8qTf+Yin+XiEdV2L/htbIoEAAxZRPgd7PT9QSjUCPDQo/wAD6CsXgm4ljvlVHgPd2Hs0y7hfppSOw+o085Zkv5cJiJ2X6p7QqirSmiq9DezFEI0VS2VMW3fFB8Rb5Id//FDIwD6dhIRlp0OE7mRjPWWPqQRGVWc6snh6d+hgs0aADAUTqNA/XRmb8wGAodMPML5nv+ggTdHXEWDDe0i5k2djwNoFkrEWjCsBK0n4TnnvkIGxJ1mSSVl75F8xSAthMydYnBgB'},
 {'type': 'tex

In [11]:
# for i, rec in get_lines("data/cli/2026-10-12.jsonl", 2, before=1, after=1):
#     print(f"--- line {i}")
#     pprint(rec, sort_dicts=False, depth=2)

In [12]:
# from llmgrab.cli.sessions import classify, session_id

# recs = [json.loads(l) for l in open("data/cli/2026-10-12.jsonl")]
# main = [r for r in recs if classify(r["request"]["body"]) == "main"]

# prev = 0
# for r in main:
#     msgs = r["request"]["body"]["messages"]
#     new = msgs[prev:]   # messages added since the previous main call
#     print(f"{len(msgs)} msgs, +{len(new)}:",
#         [(m["role"], m["content"][:40] if isinstance(m["content"], str) else [b["type"] for b in m["content"]])
#             for m in new])
#     print("  reply:", [b["type"] for b in r["response"]["body"]["content"]])
#     prev = len(msgs)

In [43]:
import json
from llmgrab.web.timeline import rpc

recs = [json.loads(l) for l in open("data/web/claude.jsonl")]
chat = [(i, r) for i, r in enumerate(recs, 1) if "/claudeai-rpc/" in r["path"]]
for i, r in chat:
    print(i, rpc(r), r["status"], r.get("duration_s"))

347 GetNewConversationDefaults 200 0.211
759 PerformAction 200 0.281
760 StreamTimeline 200 30.225
768 StreamTimeline 200 30.313
770 PerformAction 200 0.226
771 StreamTimeline 200 35.523
774 PerformAction 200 0.274
787 ReportViewing 200 0.198
806 StreamTimeline 200 None
809 PerformAction 200 0.233
810 ReportViewing 200 0.2
813 PerformAction 200 0.21
814 ReportViewing 200 0.175
816 PerformAction 200 0.373
819 ReportViewing 200 0.215
821 PerformAction 200 0.319
823 PerformAction 200 0.229
824 ReportViewing 200 0.182
831 ReportViewing 200 0.286


In [47]:
show(recs[771-1]["response"]["body"]["connect_messages"][:5])

[0]
  1:
    6: ''
[1]
  1:
    10:
      1: '1791879302113-0'
    11: 1
    1:
      1: 1
      2:
        1: '52913d6c-495d-41d9-bfb3-c27197be7e5d'
        3: 1
        4:
          1: 1791879302
          2: 48078000
        5:
          1: 1791879302
          2: 48078000
        6:
          2: 'claude-opus-5-5'
        8:

          Btype.googleapis.com/anthropic.bard.api.v1alpha.SandboxFileActivity
        10:
          1: 1
          6: 0
          7: 1
          14: 1
          15: 1
          16: 'auto'
          20: 1
        17: 4
        19: 0
        20: 1
        25: 1
        34: '0a0c1b'
        39: 419430400
        41: 0
        44: 31457280
        47: 0
        54: 4194304
        55: 4194304
      28: 'v2;lane=native;gen=f2c001fb04ac0a10'
[2]
  1:
    10:
      1: '1791879302151-0'
    11: 1
    1:
      2:
        1: '52913d6c-495d-41d9-bfb3-c27197be7e5d'
        3: 1
        4:
          1: 1791879302
          2: 48078000
        5:
          1: 1791879302
    

In [45]:
# from llmgrab.web.timeline import timeline_events
# events = timeline_events(recs)   # [(epoch_seconds, event_dict), ...]
# show(events[0][1])

In [46]:
from llmgrab.web.timeline import build_model
m = build_model(recs)
for conv in m.conversations.values():          # keys: uuid, title, model, effort, messages
    for msg in m.ordered(conv):                # role, created, done, first_text, limits, deltas...
        for b in m.message_blocks(msg):        # kind: text | thinking, text
            print(f"--- {msg['role']} / {b['kind']}")
            print(b["text"])

--- human / text
How does the claude.ai API work ? what gets sent to the anthropic endpoint from my client browser ? how is the responses parsed from my browser ? Be not too shallow and not too specific, answer only using text (no figures or artefacts)
--- assistant / thinking

--- assistant / text
Claude.ai's web app talks to Anthropic's servers through a private, undocumented API that is separate from the public developer API. The details below come from what's commonly observed in the browser and may change, but the overall architecture is fairly stable. You can check all of it yourself in your browser's DevTools Network tab while sending a message.

**Two different APIs**

The public Claude API (`api.anthropic.com/v1/messages`) is stateless. You authenticate with an API key, and every request carries the entire conversation history plus your system prompt.

Claude.ai works the other way around. Its backend is stateful: conversations, message history, files, projects and settings ar